FUNDING VS. LAYOFF SEVERITY CORRELATION
________
BUSINESS QUESTION

1. Is there a relationship between how much money a company raised (funds_raised_millions) and the PERCENTAGE of its workforce it laid off (percentage_laid_off)? 
2. Do well-funded companies cut a smaller share of staff, a larger share, or is there no real relationship?


In [1]:
import pandas as pd
import numpy as np

Loading the cleaned dataset

In [2]:
df = pd.read_csv("D:\DATA ANALYSIS\MySQL data project(A)\Data\Raw\layoff2.csv", parse_dates=["date"])

Re-add the repeat-layoff information from Analysis 1 as columns

In [3]:
df["number_of_layoff_events"] = df.groupby("company")["company"].transform("count")

df["is_repeat_layoff_company"] = df["number_of_layoff_events"] > 1

Calculating the raw correlation

In [4]:
raw_correlation = df["funds_raised_millions"].corr(df["percentage_laid_off"])
print(f"Raw correlation (funds raised vs. % laid off): {raw_correlation:.3f}")

Raw correlation (funds raised vs. % laid off): -0.066


Why we ALSO check a log-transformed version

In [5]:
df["log_funds_raised"] = np.log1p(df["funds_raised_millions"])

log_correlation = df["log_funds_raised"].corr(df["percentage_laid_off"])
print(f"Log-transformed correlation (log funds raised vs. % laid off): {log_correlation:.3f}")

Log-transformed correlation (log funds raised vs. % laid off): -0.324


Adding a readable "funding bracket" column

In [6]:
funding_bins = [-1, 50, 500, 2000, np.inf]
funding_labels = ["Small (<$50M)", "Medium ($50-500M)", "Large ($500M-2B)", "Mega (>$2B)"]

df["funding_bracket"] = pd.cut(
    df["funds_raised_millions"], bins=funding_bins, labels=funding_labels
)

Summarize average severity per funding bracket

In [10]:
bracket_summary = df.groupby("funding_bracket", observed=True).agg(
    average_percentage_laid_off=("percentage_laid_off", "mean"),
    number_of_companies=("company", "count"),
)

bracket_summary["average_percentage_laid_off"] = bracket_summary[
    "average_percentage_laid_off"
].round(3)

bracket_summary = bracket_summary.reset_index()

print("\nAverage % laid off by funding bracket:")
print(bracket_summary.to_string(index=False))


Average % laid off by funding bracket:
  funding_bracket  average_percentage_laid_off  number_of_companies
    Small (<$50M)                        0.386                  622
Medium ($50-500M)                        0.215                  967
 Large ($500M-2B)                        0.168                  314
      Mega (>$2B)                        0.160                   91


Saving the enriched master dataset

In [8]:
df.to_csv("layoffs_master_enriched.csv", index=False)

print("\nExported: layoffs_master_enriched.csv")
print(f"Columns now in the master dataset: {list(df.columns)}")


Exported: layoffs_master_enriched.csv
Columns now in the master dataset: ['company', 'location', 'industry', 'total_laid_off', 'percentage_laid_off', 'date', 'stage', 'country', 'funds_raised_millions', 'number_of_layoff_events', 'is_repeat_layoff_company', 'log_funds_raised', 'funding_bracket']
